In [ ]:
!pip install transformers datasets scikit-learn pandas numpy gradio -q

In [ ]:
import pandas as pd
import torch

train_df = pd.read_csv('train_set.csv')
val_df = pd.read_csv('validation_set.csv')

train_texts = train_df['feedback'].tolist()
train_labels = train_df['performance_class'].tolist()

val_texts = val_df['feedback'].tolist()
val_labels = val_df['performance_class'].tolist()

print(f"Loaded {len(train_texts)} training samples and {len(val_texts)} validation samples.")

Loaded 656 training samples and 116 validation samples.


In [ ]:
from transformers import AutoTokenizer

model_name = "distilbert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(model_name)

train_encodings = tokenizer(train_texts, truncation=True, padding=True, max_length=128)
val_encodings = tokenizer(val_texts, truncation=True, padding=True, max_length=128)

class EmployeeReviewDataset(torch.utils.data.Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels

    def __getitem__(self, idx):
        item = {key: torch.tensor(val[idx]) for key, val in self.encodings.items()}
        item['labels'] = torch.tensor(self.labels[idx], dtype=torch.long)
        return item

    def __len__(self):
        return len(self.labels)

train_dataset = EmployeeReviewDataset(train_encodings, train_labels)
val_dataset = EmployeeReviewDataset(val_encodings, val_labels)

In [ ]:
from transformers import AutoModelForSequenceClassification, Trainer, TrainingArguments


model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=3)

training_args = TrainingArguments(
    output_dir='./results',
    num_train_epochs=3,
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    eval_strategy="epoch",
    logging_steps=10,
    seed=42
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset
)

trainer.train()

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_projector.bias    | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
pre_classifier.weight   | MISSING    | 
classifier.weight       | MISSING    | 
classifier.bias         | MISSING    | 
pre_classifier.bias     | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(load

Epoch,Training Loss,Validation Loss


In [ ]:
import gradio as gr

def analyze_employee_sentiment(text):
    if not text.strip():
        return "Please enter valid feedback."

    inputs = tokenizer(text, return_tensors="pt", truncation=True, padding=True, max_length=128)
    inputs = {k: v.to(model.device) for k, v in inputs.items()}

    with torch.no_grad():
        outputs = model(**inputs)
        probs = torch.nn.functional.softmax(outputs.logits, dim=-1)[0]

    labels = ["Negative 🔴", "Neutral 🟡", "Positive 🟢"]
    return {labels[i]: float(probs[i]) for i in range(3)}

interface = gr.Interface(
    fn=analyze_employee_sentiment,
    inputs=gr.Textbox(lines=3, placeholder="Type employee feedback here..."),
    outputs=gr.Label(num_top_classes=3, label="Predicted Sentiment Breakdown"),
    title="Employee Performance Sentiment Analyzer",
    description="Interactive evaluation interface trained on real review datasets."
)

interface.launch(share=True)